# circTransmon — the same simulations, driven through SQDMetal directly

This notebook runs **the same five simulations** as `circTransmon_sim_functions.ipynb`, on the
**same design file**, but without the `toolkit.tools.sim` runners: every PALACE object is built,
configured and run through SQDMetal's own API, in the style of `circTransmon eigensim.ipynb`.

The only piece of the toolkit kept is `load_design` / `build_subdesign`, because the design is
stored as JSON and the sub-design construction (terminating pins left dangling by dropped
components — Ansys's `open_pins` pattern) has no SQDMetal equivalent.

**Why it exists.** The runners are convenience wrappers; this is what they expand to. Useful for
seeing exactly which SQDMetal calls produce which part of the PALACE config, for reaching options
the dataclasses do not expose, and as a reference when a run misbehaves.

**Equivalence.** The configs generated here were diffed field-by-field against the `config.json`
files that the runner versions actually wrote. Eigenmode, both capacitance runs: **identical**.
Driven modal: identical except for two keys (`AdaptiveMaxSamples`, `AdaptiveConvergenceMemory`)
that the *current* SQDMetal emits from its own defaults and the stored config predates — not a
difference between the two code paths. The last section of this notebook re-runs that diff so you
can check it yourself.

Outputs go to `./raw_sqdmetal/` so nothing here overwrites the runner results.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import gmsh
from qiskit_metal import Dict

from toolkit.tools.sim.save_load_design import load_design
from toolkit.tools.sim.sim_subdesign import build_subdesign
# A diagnostic, not a runner -- it only reads geometry, so it belongs here too.
from toolkit.tools.sim.current_loops import check_current_loops, plot_current_loops
from toolkit.tools.sim import CurrentSourceOptions

from SQDMetal.Utilities.Materials import MaterialInterface
from SQDMetal.PALACE.Eigenmode_Simulation import PALACE_Eigenmode_Simulation
from SQDMetal.PALACE.Frequency_Driven_Simulation import PALACE_Driven_Simulation
from SQDMetal.PALACE.Capacitance_Simulation import PALACE_Capacitance_Simulation
from SQDMetal.PALACE.Inductance_Simulation import PALACE_Inductance_Simulation

In [ ]:
design = load_design("circTransmon.json")

PALACE_BIN = "/Users/maximilianogatto/Library/CloudStorage/OneDrive-Personal/PhD/src/libraries/palace/build/bin/palace"
OUTPUT_ROOT = "./raw_sqdmetal/"   # trailing slash: SQDMetal warns and adds one otherwise

# Options every simulation shares. The per-simulation dicts below start from this and add the
# keys that only their solver understands -- SQDMetal fills anything omitted from the class's
# own `default_user_options`.
BASE_OPTIONS = {
    "mesh_refinement": 0,             # global refinement passes (0 = coarsest)
    "dielectric_material": "silicon", # must match design.chips.main.material
    "fillet_resolution": 12,          # segments per quarter turn on a filleted path
    "palace_dir": PALACE_BIN,
}

# The CPW path refinement reused by most runs: fine on the trace, tapering outwards.
CPW_MESH = dict(dist_resolution=10e-6, min_size=6e-6, max_size=150e-6, taper_dist_min=10e-6)

def reset_gmsh():
    """SQDMetal leaves the gmsh session initialised after prepare_simulation(); the next sim's
    GMSH_Geometry_Builder expects to open a fresh one. The runners do this in their cleanup
    context manager -- here it has to be explicit, so call it before building a new simulation."""
    if gmsh.isInitialized():
        gmsh.finalize()

## Eigenmode

Bare resonator: `meandro_1` + `stg1` only, with the pin that used to reach `Q1` terminated open.
No port — an eigenmode solve has no excitation, it looks for the source-free modes.

In [ ]:
reset_gmsh()

design_eigen = build_subdesign(
    design,
    keep=['meandro_1', 'stg1'],
    terminations={('meandro_1', 'end'): 'open'},   # used to connect to Q1, which is dropped
)

eigen_options = {
    **BASE_OPTIONS,
    "starting_freq": 5e9,     # Hz -- SLEPc shift-invert starts its search here
    "number_of_freqs": 1,     # how many eigenmodes to look for
    "solns_to_save": 1,       # field solutions written for ParaView
    "solver_order": 1,        # finite-element order
    "solver_tol": 1.0e-8,
    "solver_maxits": 200,
    "num_cpus": 2,
}

eigen_sim = PALACE_Eigenmode_Simulation(
    name='resonator_eigensim', metal_design=design_eigen,
    sim_parent_directory=OUTPUT_ROOT, mode='simPC', meshing='GMSH',
    user_options=eigen_options, create_files=True)

eigen_sim.add_metallic(1)
eigen_sim.add_ground_plane()

# NOTE: the argument order here is NOT the order the runner version passes them in -- see the
# verification section at the bottom. Each interface must get its own material:
#   substrate_air   -> the exposed silicon surface        -> 'Silicon-Vacuum'
#   substrate_metal -> silicon underneath the aluminium   -> 'Silicon-Aluminium'
#   metal_air       -> the aluminium surface              -> 'Aluminium-Vacuum'
# Needed for Q; without it you get frequencies only.
eigen_sim.setup_EPR_interfaces(
    substrate_air=MaterialInterface('Silicon-Vacuum'),
    substrate_metal=MaterialInterface('Silicon-Aluminium'),
    metal_air=MaterialInterface('Aluminium-Vacuum'))

eigen_sim.fine_mesh_along_path(qObjName='meandro_1', **CPW_MESH)

eigen_sim.enable_mesh_refinement(
    5,                             # AMR passes
    max_DoFs=0,                    # 0 = no cap
    tolerance=1e-2,                # stop once the error estimate drops below this
    Dorfler_marking_fraction=0.3,  # fraction of elements refined per pass
    save_iterations_data=False, save_iterations_mesh=False, nonconformal=True)

eigen_sim.prepare_simulation()     # writes the .msh and the .json

In [ ]:
eigen_data = eigen_sim.run(stream_output=True)

# eig.csv columns: [m, Re{f} (GHz), Im{f} (GHz), Q, err_bkwd, err_abs] -- GHz, and column 2 is
# the linewidth, not the solver error.
for row in eigen_data:
    print(f"Mode {int(row[0])}: {row[1]:.6f} GHz, Im(f) = {row[2]*1e3:.3g} MHz, "
          f"Q = {row[3]:.4g}, abs. error = {row[5]:.2e}")

eigenfreq = eigen_data[0][1] * 1e9  # Hz

## Driven modal

Full-wave sweep with lumped ports on both launchpads. `set_port_excitation` takes the port's
**1-indexed creation order**, so `LP1` created first is port 1.

Note `solns_to_save` and the sweep are set independently: the runner version left
`solns_to_save = 1001` from the first coarse sweep while re-running the fine one with 4001
points, which is why the stored config shows `SaveStep: 1001`. Reproduced here for the
equivalence check.

In [ ]:
reset_gmsh()

f_center, f_bandwidth, n_freq = eigenfreq, 1e9, 1001

design_driven = build_subdesign(
    design,
    keep=['meandro_1', 'stg1', 'linea_1', 'LP1', 'LP2'],
    terminations={('meandro_1', 'end'): 'open'},
)

driven_options = {
    **BASE_OPTIONS,
    "solns_to_save": n_freq,
    "solver_order": 1,
    "solver_tol": 1.0e-8,
    "solver_maxits": 300,
    "adaptive_solver_tol": 1e5,   # PROM tolerance; the adaptive solver can struggle on high Q
    "num_cpus": 4,
}

driven_sim = PALACE_Driven_Simulation(
    name='resonator_drivensim', metal_design=design_driven,
    sim_parent_directory=OUTPUT_ROOT, mode='simPC', meshing='GMSH',
    user_options=driven_options, create_files=True)

driven_sim.add_metallic(1)
driven_sim.add_ground_plane()

driven_sim.create_port_CPW_on_Launcher('LP1', len_launch=20e-6,
                                       impedance_R=50, impedance_L=0, impedance_C=0)
driven_sim.create_port_CPW_on_Launcher('LP2', len_launch=20e-6,
                                       impedance_R=50, impedance_L=0, impedance_C=0)
driven_sim.set_port_excitation(port_index=1)   # LP1 drives, LP2 is passive -> S21

driven_sim.fine_mesh_along_path(qObjName='meandro_1', **CPW_MESH)
driven_sim.fine_mesh_along_path(qObjName='linea_1', **CPW_MESH)

# The runners take (center, bandwidth, n_points); SQDMetal takes (start, end, step).
driven_sim.set_freq_values(freq_start=f_center - f_bandwidth/2,
                           freq_end=f_center + f_bandwidth/2,
                           freq_step=f_bandwidth/(n_freq - 1))

driven_sim.prepare_simulation()

In [ ]:
driven_result = driven_sim.run(stream_output=True)
print(driven_result.keys())

In [ ]:
freq, s21 = driven_result['freqs'], driven_result['S21']
mag_s21 = 20*np.log10(np.abs(s21))
i_min = np.argmin(mag_s21)
print(f"S21 minimum {mag_s21[i_min]:.2f} dB at {freq[i_min]/1e9:.4f} GHz")

fig, ax = plt.subplots(2, 1, sharex=True, figsize=(11, 6))
ax[0].plot(freq/1e9, mag_s21);              ax[0].set_ylabel('|S21| [dB]')
ax[1].plot(freq/1e9, np.angle(s21, deg=True)); ax[1].set_ylabel('phase [deg]')
ax[1].set_xlabel('frequency [GHz]')
ax[0].axvline(freq[i_min]/1e9, color='r', ls='--')
plt.show()

## Capacitance

Two electrostatic runs. Both ends of the meander are terminated **open** here, on purpose: an
isolated island is what makes a conductor a well-defined node for the capacitance matrix. (That
is the opposite of what the inductance run needs — see the note there.)

`run_capacitance_sim` was called twice with the same `cap_options`, so both runs share one
options dict here too.

In [ ]:
cap_options = {
    **BASE_OPTIONS,
    "solns_to_save": -1,      # -1 = one per conductor
    "solver_order": 2,
    "solver_tol": 1.0e-8,
    "solver_maxits": 100,
    "num_cpus": 4,
}

def build_capacitance_sim(name, keep, terminations, setup_mesh):
    """Everything the two capacitance runs have in common."""
    reset_gmsh()
    sub = build_subdesign(design, keep=keep, terminations=terminations)
    sim = PALACE_Capacitance_Simulation(
        name=name, metal_design=sub, sim_parent_directory=OUTPUT_ROOT,
        mode='simPC', meshing='GMSH', user_options=cap_options, create_files=True)
    sim.add_metallic(1)
    sim.add_ground_plane()
    setup_mesh(sim)
    sim.prepare_simulation()
    return sim

### Resonator–feedline coupling capacitance

In [ ]:
def mesh_resonator_and_feedline(sim):
    sim.fine_mesh_along_path(qObjName='meandro_1', **CPW_MESH)
    sim.fine_mesh_along_path(qObjName='linea_1', **CPW_MESH)

cap_sim_res = build_capacitance_sim(
    'resonator_capacitance',
    keep=['meandro_1', 'linea_1', 'LP1', 'LP2'],
    terminations={('meandro_1', 'end'): 'open', ('meandro_1', 'start'): 'open'},
    setup_mesh=mesh_resonator_and_feedline)

resonator_cap_res = cap_sim_res.run(stream_output=True)
display(resonator_cap_res)

In [ ]:
# terminal-C.csv is the MAXWELL capacitance matrix: positive diagonal, negative off-diagonal.
# The physical (SPICE) capacitance between two conductors is -C_ij, hence the abs().
coupling_cap = abs(resonator_cap_res['Cond3']["launch_pad_LP1"])
print(f"Coupling capacitance: {coupling_cap:.2f} fF")

cap_sim_res.display_conductor_indices()   # which conductor is which row/column
plt.show()

### Qubit capacitance

In [ ]:
cap_sim_qubit = build_capacitance_sim(
    'qubit_capacitance',
    keep=['Q1'],
    terminations={('Q1', 'claw_a'): 'open', ('Q1', 'charge_line'): 'open'},
    setup_mesh=lambda sim: sim.fine_mesh_components(
        ['Q1'], min_size=6e-6, max_size=100e-6, taper_dist_min=10e-6, metals_only=False))

qubit_cap_res = cap_sim_qubit.run(stream_output=True)
display(qubit_cap_res)

Cq  = abs(qubit_cap_res['ground_main_plane']["pads_Q1"])
Ccl = abs(qubit_cap_res['charge_line_Q1']["pads_Q1"])
Cg  = abs(qubit_cap_res['claw_a_Q1']["pads_Q1"])
print(f"Qubit to ground:      {Cq:.2f} fF")
print(f"Qubit to claw:        {Cg:.2f} fF")
print(f"Qubit to charge line: {Ccl:.2f} fF")

## Inductance (magnetostatic)

**This section deliberately does NOT reproduce the stored inductance config.** That run was
broken: both of its linear solves diverged (`norm(Ax-b)/norm(b) = 2.456e+01` and `1.507e-01`
in its `out.log`) because the feedline terminal did not cut a closed current loop. What follows
is the corrected setup, matching the current `circTransmon_sim_functions.ipynb`.

Every current-source terminal must be a **cut in a galvanically closed loop** — inject current
at the cut and it has to be able to return. So each conductor needs a short to ground at one end
and the terminal at the other:

* **resonator** — already shorted by `stg1`, so the terminal goes on the free end.
* **feedline** — `LP1—linea_1—LP2` is one isolated island, so `LP2` is dropped and
  `linea_1.end` terminated `'short'` (`ShortToGround` draws no metal; it just lets the route's
  ground cut stop there, so the trace tip touches the plane).

This is the exact opposite of the capacitance section, where both ends had to be *open*.

In [ ]:
# PRE-FLIGHT. Take the metal, add the U-clips, remove the terminal patches, count the connected
# pieces: 1 means every cut has a way around through the metal, 2+ means some conductor is an
# island reachable only through its own cut -- div J != 0, no solution, and PALACE would still
# write a full terminal-M.csv of garbage. Two seconds against three minutes of doomed solve.
IND_KEEP = ['meandro_1', 'stg1', 'linea_1', 'LP1']          # LP2 dropped on purpose
IND_TERMINATIONS = {('linea_1', 'end'): 'short',            # closes the feedline loop
                    ('meandro_1', 'end'): 'open'}           # gap for the U-clip terminal
UCLIP = dict(thickness_side=20e-6, thickness_back=20e-6, separation_gap=20e-6)

# Same geometry the raw create_current_source_with_Uclip_* calls below produce -- the checker
# uses SQDMetal's own U-clip helpers, so what it draws is what the mesher will build.
IND_SOURCES = [
    CurrentSourceOptions(component_name='LP1', type='Uclip_on_Launcher', **UCLIP),
    CurrentSourceOptions(component_name='meandro_1', type='Uclip_on_Route', pin_name='end', **UCLIP),
]

loops = check_current_loops(design, IND_SOURCES, components=IND_KEEP,
                            terminations=IND_TERMINATIONS, strict=True)
print(loops.summary())
TERMINAL_NAMES = loops.terminal_names   # rows/columns of terminal-M.csv, in source order

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
plot_current_loops(loops, ax=axes[0])              # whole chip
plot_current_loops(loops, ax=axes[1], zoom=0)      # terminal 1, at LP1
plot_current_loops(loops, ax=axes[2], zoom=1)      # terminal 2, at meandro_1.end
fig.tight_layout()
plt.show()

In [ ]:
reset_gmsh()

design_ind = build_subdesign(design, keep=IND_KEEP, terminations=IND_TERMINATIONS)

ind_options = {
    **BASE_OPTIONS,
    "solns_to_save": -1,      # -1 = one per terminal
    "solver_order": 2,
    "solver_tol": 1.0e-8,
    "solver_maxits": 5000,    # AMS on thin PEC sheets converges slowly; 200 is nowhere near enough
    "num_cpus": 4,
}

mag_sim = PALACE_Inductance_Simulation(
    name='resonator_inductance', metal_design=design_ind,
    sim_parent_directory=OUTPUT_ROOT, mode='simPC', meshing='GMSH',
    user_options=ind_options, create_files=True)

mag_sim.add_metallic(1)
mag_sim.add_ground_plane()

# Terminal 1 and 2, in this order -- the rows/columns of terminal-M.csv follow it.
mag_sim.create_current_source_with_Uclip_on_Launcher('LP1', **UCLIP)
mag_sim.create_current_source_with_Uclip_on_Route('meandro_1', 'end', **UCLIP)

mag_sim.fine_mesh_along_path(qObjName='meandro_1', **{**CPW_MESH, 'min_size': 4e-6})
mag_sim.fine_mesh_along_path(qObjName='linea_1', **{**CPW_MESH, 'min_size': 4e-6})

mag_sim.prepare_simulation()

In [ ]:
# SQDMetal's run() ends by calling retrieve_data(), which reads surface-F.csv -- a file PALACE
# only writes when the config declares SurfaceFlux post-processing, i.e. when integration areas
# were added. There are none here (the inductance matrix needs none), so on an unpatched SQDMetal
# that read raises even though the solve finished and every result file is already on disk.
try:
    mag_sim.run(stream_output=True)
except FileNotFoundError as err:
    if 'surface-F.csv' not in str(err):
        raise
    print("PALACE finished; skipped SQDMetal's flux read (no integration areas declared)")

# PALACE only *warns* when a solve stalls and still writes a full terminal-M.csv, so check.
log = Path(mag_sim._output_data_dir) / 'out.log'
stalled = [l.strip() for l in log.read_text(errors='replace').splitlines()
           if 'did not converge' in l]
if stalled:
    raise RuntimeError("PALACE did not converge:\n  " + "\n  ".join(stalled))
print("all linear solves converged")

raw = mag_sim.retrieve_magnetostatic_data()
M_mat = raw['inductance_matrix']

In [ ]:
import pandas as pd
from scipy.special import ellipk
import shapely

display(pd.DataFrame(M_mat*1e12, index=TERMINAL_NAMES, columns=TERMINAL_NAMES).round(3))  # pH
L_feed, L_res, M_dc = M_mat[0, 0], M_mat[1, 1], M_mat[0, 1]

# Closed-form CPW inductance per length as a reference: k = w/(w+2g), L' = (mu0/4) K(k')/K(k).
# scipy's ellipk takes the PARAMETER m = k^2.
w, g = 20e-6, 11e-6
k2 = (w/(w + 2*g))**2
Lp = np.pi*1e-7 * ellipk(1 - k2)/ellipk(k2)
length = lambda c: shapely.LineString(np.array(design.components[c].get_points())*1e-3).length

print(f"L feedline  {L_feed*1e9:8.3f} nH   (CPW estimate {Lp*length('linea_1')*1e9:.2f} nH)")
print(f"L resonator {L_res*1e9:8.3f} nH   (CPW estimate {Lp*length('meandro_1')*1e9:.2f} nH)")
print(f"M (DC)      {M_dc*1e12:8.3f} pH   ->  k = {abs(M_dc)/np.sqrt(L_feed*L_res):.2e}")

# Cauchy-Schwarz on the curl-curl inner product: k > 1 means what came back is not a field.
assert abs(M_dc) < np.sqrt(L_feed*L_res), "|M| > sqrt(L1 L2): the run is broken."